# 2.3 特征工程

## 学习目标
- 从 OHLCV 原始数据构建有价值的特征
- 掌握滚动统计量、滞后特征的创建
- 理解特征标准化与避免未来数据泄露（Look-ahead Bias）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns

# 下载数据
raw = yf.download('AAPL', start='2020-01-01', end='2024-01-01', progress=False)
df = pd.DataFrame({
    'Open': raw['Open'].squeeze(),
    'High': raw['High'].squeeze(),
    'Low': raw['Low'].squeeze(),
    'Close': raw['Close'].squeeze(),
    'Volume': raw['Volume'].squeeze(),
})
print(f'原始数据: {df.shape}')
df.head(3)

## 1. 收益率特征

In [ ]:
# 不同周期收益率
df['ret_1d'] = df['Close'].pct_change(1)    # 1日收益
df['ret_5d'] = df['Close'].pct_change(5)    # 5日（周）收益
df['ret_20d'] = df['Close'].pct_change(20)  # 20日（月）收益

# 对数收益率
df['log_ret_1d'] = np.log(df['Close'] / df['Close'].shift(1))

# 日内价格特征
df['daily_range'] = (df['High'] - df['Low']) / df['Close']  # 日内振幅
df['gap'] = (df['Open'] - df['Close'].shift(1)) / df['Close'].shift(1)  # 跳空缺口

print('新增收益率特征：')
print(df[['ret_1d', 'ret_5d', 'ret_20d', 'daily_range', 'gap']].describe().round(4))

## 2. 滚动统计量特征

In [ ]:
# 滚动均值
for w in [5, 10, 20, 60]:
    df[f'ma_{w}'] = df['Close'].rolling(w).mean()
    df[f'vol_{w}d'] = df['ret_1d'].rolling(w).std() * np.sqrt(252)  # 年化波动率

# 价格与均线比值（相对位置）
df['price_ma20_ratio'] = df['Close'] / df['ma_20'] - 1

# 成交量变化
df['volume_ma20'] = df['Volume'].rolling(20).mean()
df['volume_ratio'] = df['Volume'] / df['volume_ma20']  # 量比

print('滚动特征已添加')

# 可视化价格与移动均线
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['Close'], label='收盘价', linewidth=1, alpha=0.8)
for w, color in zip([20, 60], ['orange', 'red']):
    ax.plot(df.index, df[f'ma_{w}'], label=f'MA{w}', linewidth=1.5, color=color)
ax.set_title('AAPL 价格 + 移动均线', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 滞后特征与未来标签

> ⚠️ **Look-ahead Bias（未来数据泄露）是量化回测中最常见的陷阱！**
> 
> 预测 $t$ 时刻的特征，**只能**使用 $t-1$ 或更早的数据。

In [ ]:
# 正确做法：使用滞后特征
for lag in [1, 2, 3, 5]:
    df[f'ret_lag_{lag}'] = df['ret_1d'].shift(lag)  # shift(1) 表示用昨日数据

# 构造预测标签：明天的涨跌方向
df['target_direction'] = (df['ret_1d'].shift(-1) > 0).astype(int)  # 1=明天涨, 0=跌
df['target_ret'] = df['ret_1d'].shift(-1)  # 明天的实际收益率

# ⚠️ 注意：构建特征时 shift(-1) 只能用于标签，不能用于特征！
print('特征矩阵示例：（最后几行因 shift(-1) 存在 NaN）')
feature_cols = ['ret_1d', 'ret_lag_1', 'price_ma20_ratio', 'vol_20d', 'volume_ratio']
df[feature_cols + ['target_direction']].dropna().tail(5)

## 4. 特征相关性分析

In [ ]:
feat_matrix = df[['ret_1d', 'ret_5d', 'ret_20d', 'daily_range',
                   'vol_20d', 'volume_ratio', 'price_ma20_ratio',
                   'target_ret']].dropna()

corr = feat_matrix.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('特征相关性矩阵', fontsize=13)
plt.tight_layout()
plt.show()

# 与目标变量相关性
print('\n各特征与明日收益的相关性：')
print(corr['target_ret'].drop('target_ret').sort_values(key=abs, ascending=False))

## 5. 特征标准化

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

feature_cols = ['ret_1d', 'ret_5d', 'daily_range', 'vol_20d', 'volume_ratio']
X = feat_matrix[feature_cols].dropna()

# Z-score 标准化
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols, index=X.index)

print('标准化后的统计量（均值≈0，标准差≈1）：')
print(X_scaled.describe().round(3))

## 🎯 练习

1. 新增特征：高低价比 `(High - Low) / Open`，分析其与次日收益率的相关性。
2. 尝试构造**周频**特征：每周五收盘时，用过去4周数据计算特征。
3. 思考：为什么我们不能用**未来的成交量**来预测今天的价格方向？

---
**下一模块** → `../03_indicators/01_trend_indicators.ipynb`